In [ ]:
try:
    import tensorflow_datasets as tfds 
except:
    !pip install tfds-nightly tensorflow-cpu
    import tfds

In [ ]:
from pathlib import Path
from src.flow_models.config import Config
from src.flow_models.trainer_gen import GenerationTrainer
import argparse
import sys

from functools import partial

import matplotlib.pyplot as plt

from jax import config
from jax import nn, vmap
import jax.numpy as jnp
import jax.random as jr

import jax_dataloader as jdl
config.update('jax_cuda_visible_devices', '1')

class Args(argparse.Namespace):
  data_path = None
  model_type = 'fm_mix'
  input_shape = None
  output_shape = None
  latent_shape = None
  vae_weight = 0.0
  recon_weight = 1.0
  reg_weight = 0.0
  crn_type = None
  network_type = None
  hidden_dims = None
  recon_loss_type = 'mse'
  encoder_model_type = 'identity' # 'mlp', 'mlp_normal', 'resnet', 'resnet_normal', 'identity', 'linear'
  decoder_model_type = 'identity' # 'mlp', 'resnet', 'identity'
  decoder_type = 'linear'
  no_noise_schedule = True

args = Args()
key = jr.PRNGKey(137)

In [ ]:
# Construct a tf.data.Dataset
test_ds = tfds.load('mnist', split='test', as_supervised=True)
train_ds = tfds.load('mnist', split='train', as_supervised=True)

jds = jdl.DataLoader(train_ds, 'tensorflow', batch_size=len(train_ds), shuffle=True)
x_train, y_train = next(iter(jds))

x_train = x_train / 256
y_train = nn.one_hot(y_train, 10)

print(x_train.shape, y_train.shape)

jds = jdl.DataLoader(test_ds, 'tensorflow', batch_size=len(test_ds), shuffle=True)
x_test, y_test = next(iter(jds))

x_test = x_test / 256
y_test = nn.one_hot(y_test, 10)
print(x_test.shape, y_test.shape)


def image_to_pointcloud(
    key, x, patch_grid, num_samples=100, image_size=28
):  

    batch_shape = len(x)
    probs = x.mean(-1).reshape(batch_shape, -1)
    probs /= probs.sum(-1, keepdims=True)
    pixel_args = jr.categorical(key, jnp.log(probs), shape=(num_samples, batch_shape), replace=False)
    idxs = patch_grid[pixel_args]
    locs = 5 * (2 * (idxs + 0.5) / image_size - 1)
    return jnp.moveaxis(locs, 0, 1)

image_size = 28
patch_size = 1
patch_grid_x, patch_grid_y = jnp.meshgrid(
    jnp.arange(image_size // patch_size),
    jnp.arange(image_size // patch_size),
    indexing="ij",
)
patch_grid = jnp.stack(
        [patch_grid_x.reshape(-1), patch_grid_y.reshape(-1)], -1
    )

num_samples = 32
x_pc = image_to_pointcloud(key, x_train[:64], patch_grid, num_samples=num_samples)

In [ ]:
fig, axes = plt.subplots(2, 4)

for i in range(4):
    axes[0, i].imshow(x_train[i])
    axes[1, i].scatter(x_pc[i, ..., 1], - x_pc[i, ..., 0])
    axes[1, i].set_ylim([-5, 5])
    axes[1, i].set_xlim([-5., 5.])

fig.tight_layout()

In [ ]:
config_file = 'examples/mnist/pointcloud_config.yaml'
config_path = Path(config_file)
loaded_config = Config.load_yaml(config_file)
base_config = Config.merge_with_defaults(loaded_config)

y_sample = image_to_pointcloud(key, x_train[0:1], patch_grid, num_samples=num_samples).reshape(-1, 2)
x_sample = y_train[0:1]

def make_trainer(args, seed):
    config = base_config.override_from_args(args, args.model_type, False)

    trainer = GenerationTrainer(
        config=config,
        learning_rate=5e-3,
        optimizer_name='adamw',
        seed=seed,
        unconditional=False,
        warmup_steps=0,
        model_type=args.model_type
    )

    trainer.initialize(x_sample, y_sample)

    return trainer

def run_experiment(key, args, x_train, y_train, x_test, y_test, num_epochs=50):
    fig, axes = plt.subplots(2, 6, figsize=(16, 8), sharex='col', sharey=False)
    for i, ns in enumerate(['linear']):
        for j, learn in enumerate([False]):
            args.noise_schedule = ns
            args.noise_schedule_learnable = learn

            key, _key = jr.split(key)

            trainer = make_trainer(args, _key[0].item())

            history = trainer.train(
                x_data=x_train,
                y_data=y_train,
                num_epochs=num_epochs,
                batch_size=5120,
                validation_data=(x_test, y_test),
                dropout_epochs=0
            )

            key, _key = jr.split(key)
            x_gen = jnp.eye(10)[:, None].repeat(128, axis=1).reshape(-1, 10)
            y_gen = trainer.conditional_generate(x_gen, num_steps=200, prng_key=_key).reshape(10, 128, -1)

    axes[0, 0].plot(jnp.arange(1, num_epochs + 1), history['train_losses'], label='train')
    axes[1, 0].plot(jnp.arange(1, num_epochs + 1), history['val_losses'], label='val')

    for i, ax in enumerate(axes[:, 1:].flatten()):
        ax.scatter(y_gen[i, :, 1], - y_gen[i, :, 0])
        ax.set_xlim([-5, 5])
        ax.set_ylim([-5, 5])
        ax.set_title(f'Number {i}')

    axes[1, 0].set_xlabel('Epoch')

    fig.tight_layout()
    return key

In [ ]:
key, _key = jr.split(key)
train_y = image_to_pointcloud(_key, x_train, patch_grid, num_samples=num_samples).reshape(-1, 2)
train_x = y_train[:, None].repeat(num_samples, axis=1).reshape(-1, 10)

test_x = y_test[:, None].repeat(num_samples, axis=1).reshape(-1, 10)
test_y = image_to_pointcloud(_key, x_test, patch_grid, num_samples=num_samples).reshape(-1, 2)
key = run_experiment(key, args, train_x, train_y, test_x, test_y, num_epochs=200)